# SkyGuard AI Sandbox

All imports + data + trained models ready. Run the cells top to bottom, then test any layer below.

**Kernel:** the project venv (`python3.11`, already has numpy/pandas/sklearn/shap/chronos).
**Working dir:** this notebook lives in `microservices/app/core/`, so relative imports below resolve to `microservices/app/`.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# In a notebook there is no __file__ — hardcode the app dir instead.
APP_DIR = Path.cwd().parent  # microservices/app
sys.path.insert(0, str(APP_DIR))

# Core config (dataset choice, limits)
sys.path.insert(0, str(Path.cwd()))
import config

# All layers (plain dict payloads, see AGENTS.md contract)
from layers_v2.physics import evaluate_physics
from layers_v2.gap import detect_gaps
from layers_v2.ml import FEATURES, add_features, isolation_forest_shap, train_ml_model
from layers_v2.frozen import evaluate_frozen
from layers_v2.forecasting import forecast_reading
from layers_v2.spatial import evaluate_spatial
from layers_v2.fusion import fuse
from layers_v2.aging import evaluate_aging

print("imports OK")

imports OK


/Users/lakshya/Personal/Coding/Sky-Guard-AI/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
# Load the two datasets + station metadata
clean_df = pd.read_parquet(config.CLEAN_PARQUET)
eval_df = pd.read_parquet(config.EVAL_PARQUET)

STATION_COLS = ["lat", "lon", "elevation_m"]

def reading_from_row(row: pd.Series) -> dict:
    """DataFrame row -> L1-style reading dict (AGENTS.md contract)."""
    return {
        "timestamp": str(row["timestamp"]),
        "station_id": row["station_id"],
        "temp_c": row["temp_c"],
        "pressure_hpa": row["pressure_hpa"],
        "humidity_pct": row["humidity_pct"],
    }

def station_from_row(row: pd.Series) -> dict:
    return {col: row[col] for col in STATION_COLS}

# Sorted per-station copies for window-based layers (L3/L4/aging)
clean_df = clean_df.sort_values(["station_id", "timestamp"]).reset_index(drop=True)
eval_df = eval_df.sort_values(["station_id", "timestamp"]).reset_index(drop=True)

print("datasets OK:", len(clean_df), "clean /", len(eval_df), "eval rows")


datasets OK: 263160 clean / 263160 eval rows


In [9]:
# Train the L2 IsolationForest + SHAP explainer on the clean baseline
clean_readings = [reading_from_row(r) for _, r in clean_df.iterrows()]
model, explainer = train_ml_model(clean_readings)
print("L2 model + explainer ready")

L2 model + explainer ready


In [11]:
# Optional: load the Chronos-2 forecasting pipeline (downloads ~1-2 GB on first run)
from chronos import Chronos2Pipeline
pipeline = Chronos2Pipeline.from_pretrained("amazon/chronos-2")
print("Chronos-2 ready")

Loading weights: 100%|██████████| 170/170 [00:00<00:00, 4781.89it/s]

Chronos-2 ready


## Example: one station, single-reading layers (L1, L2 ML, L2 gap)

Grab a station slice and run the per-reading layers.

In [13]:
station = "HYD001"
sdf = eval_df[eval_df["station_id"] == station].head(500000).reset_index(drop=True)
readings = [reading_from_row(r) for _, r in sdf.iterrows()]

l1 = [evaluate_physics(r, station=station_from_row(sdf.iloc[i])) for i, r in enumerate(readings)]
l2 = isolation_forest_shap(readings, model, explainer, FEATURES)
gap = detect_gaps(readings)

print("L1 flagged:", sum(1 for p in l1 if p["predicted_anomaly"]))
print("L2 ML flagged:", sum(1 for p in l2 if p["predicted_anomaly"]))
print("L2 gap flagged:", sum(1 for p in gap if p["predicted_anomaly"]))
print()
for i, p in enumerate(l2):
    if p["predicted_anomaly"]:
        print(i, readings[i]["timestamp"], p["affected_sensors"], p["reason"])

L1 flagged: 100
L2 ML flagged: 0
L2 gap flagged: 12



## Example: window-based layers (L3 frozen, L4 forecast, aging)

These need a window of chronological readings for ONE station.

In [14]:
# L3 frozen: sliding window over one station
from layers_v2.frozen import MIN_WINDOW_LEN

win_readings = [reading_from_row(r) for _, r in sdf.iloc[:MIN_WINDOW_LEN + 1].iterrows()]
print("L3 frozen:", evaluate_frozen(win_readings)["reason"])

# Aging: needs months of history
long_sdf = eval_df[eval_df["station_id"] == station].reset_index(drop=True)
long_readings = [reading_from_row(r) for _, r in long_sdf.iterrows()]
print("Aging:", evaluate_aging(long_readings)["reason"])

L3 frozen: None
Aging: Sensor(s) showing significant drift: pressure_hpa, temp_c


## Example: L5 spatial + L6 fusion

L5 needs ALL stations at one timestamp; L6 fuses everything into a verdict.

In [15]:
# L5: one timestamp, all stations
ts = eval_df["timestamp"].min()
snap = eval_df[eval_df["timestamp"] == ts].reset_index(drop=True)
stations = {r["station_id"]: station_from_row(r) for _, r in snap.iterrows()}
snap_readings = [reading_from_row(r) for _, r in snap.iterrows()]
l5 = evaluate_spatial(snap_readings, stations)
for p in l5:
    s = p["checks"]["spatial"]
    if s.get("evaluable"):
        print(f"{p['station_id']}: temp_z={s['temp_z']:>6} pressure_z={s['pressure_z']:>6} -> {p['reason']}")

# L6: fuse a small batch (L1+L2+L5)
l1_batch = [evaluate_physics(r, station=stations[r["station_id"]]) for r in snap_readings]
l2_batch = isolation_forest_shap(snap_readings, model, explainer, FEATURES)
verdicts = fuse(l1=l1_batch, l2=l2_batch, l5=l5)
print()
for v in verdicts:
    f = v["checks"]["fusion"]
    if f["verdict"] != "healthy":
        print(f"{v['station_id']}: {f['verdict']} (conf={f['confidence']}) {f['contributing_layers']}")

BLR001: temp_z=-11.767 pressure_z=  9.41 -> 2 neighbours: temp_z=-11.77 (disagree), pressure_z=9.41 (disagree)
BOM001: temp_z=-1.149 pressure_z= -1.26 -> 2 neighbours: temp_z=-1.15 (agree), pressure_z=-1.26 (agree)
BOM002: temp_z=-0.324 pressure_z=-0.266 -> 2 neighbours: temp_z=-0.32 (agree), pressure_z=-0.27 (agree)
CCU001: temp_z= 4.344 pressure_z=-14.24 -> 2 neighbours: temp_z=4.34 (disagree), pressure_z=-14.24 (disagree)
COK001: temp_z= 0.528 pressure_z=-0.883 -> 2 neighbours: temp_z=0.53 (agree), pressure_z=-0.88 (agree)
DEL001: temp_z=-1.173 pressure_z=-0.669 -> 4 neighbours: temp_z=-1.17 (agree), pressure_z=-0.67 (agree)
DEL002: temp_z=-2.471 pressure_z= 0.697 -> 4 neighbours: temp_z=-2.47 (agree), pressure_z=0.70 (agree)
GAU001: temp_z=-0.312 pressure_z= 0.552 -> 2 neighbours: temp_z=-0.31 (agree), pressure_z=0.55 (agree)
HYD001: temp_z= 7.919 pressure_z=-0.746 -> 2 neighbours: temp_z=7.92 (disagree), pressure_z=-0.75 (agree)
JAI001: temp_z= 0.415 pressure_z=-0.238 -> 3 neighbo